In [1]:
import pickle
from pathlib import Path
 
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import OneHotEncoder

In [2]:
HORIZONS = [7, 30, 60, 90]
LAGS = [1, 7, 30, 60, 90]
ROLLING_WINDOWS = [7, 30]
 
CATEGORICAL_FEATURES = [
    "route_id",
    "origin",
    "destination_port",
    "vessel_class",
    "cargo_type",
]
 
XGB_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

In [3]:
def get_project_paths() -> dict:
    project_root = Path.cwd().parent
    paths = {
        "root": project_root,
        "raw": project_root / "data" / "raw",
        "processed": project_root / "data" / "processed",
        "models": project_root / "models",
        "results": project_root / "results",
    }
    for key in ("processed", "models", "results"):
        paths[key].mkdir(parents=True, exist_ok=True)
    return paths

In [4]:
def load_data() -> pd.DataFrame:
    project_root = Path.cwd().parent
    raw_data = project_root / "data" / "raw"
 
    freight = pd.read_csv(raw_data / "freight_rates_daily.csv")
    freight["date"] = pd.to_datetime(freight["date"])
 
    print(
        f"Freight data: {freight.shape} | "
        f"{freight['route_id'].nunique()} routes | "
        f"{freight['date'].min().date()} -> {freight['date'].max().date()}"
    )
    return freight

In [5]:
def engineer_features(freight: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str]]:
    model_data = freight.sort_values(["route_id", "date"]).reset_index(drop=True)
 
    for h in HORIZONS:
        model_data[f"target_{h}d"] = (
            model_data.groupby("route_id")["freight_usd_mt"].shift(-h)
        )
 
    for lag in LAGS:
        model_data[f"freight_lag_{lag}d"] = (
            model_data.groupby("route_id")["freight_usd_mt"].shift(lag)
        )
 
    for w in ROLLING_WINDOWS:
        model_data[f"freight_rolling_mean_{w}d"] = (
            model_data.groupby("route_id")["freight_usd_mt"]
            .transform(lambda x: x.shift(1).rolling(w).mean())
        )
 
    model_data["year"] = model_data["date"].dt.year
    model_data["month"] = model_data["date"].dt.month
    model_data["day_of_week"] = model_data["date"].dt.dayofweek
    model_data["day_of_year"] = model_data["date"].dt.dayofyear
 
    history_features = [f"freight_lag_{l}d" for l in LAGS] + [
        f"freight_rolling_mean_{w}d" for w in ROLLING_WINDOWS
    ]
    numerical_features = [
        "distance_nm",
        "synthetic_bunker_price_usd_mt",
        "freight_usd_mt",
        *history_features,
        "year",
        "month",
        "day_of_week",
        "day_of_year",
    ]
    return model_data, history_features, numerical_features

In [6]:
def split_train_val_test(model_data: pd.DataFrame, history_features: list[str]):
    train = model_data[model_data["date"] < "2025-01-01"].copy()
    validation = model_data[
        (model_data["date"] >= "2025-01-01") & (model_data["date"] < "2025-10-01")
    ].copy()
    test = model_data[model_data["date"] >= "2025-10-01"].copy()
 
    print(f"Train: {train.shape} | Validation: {validation.shape} | Test: {test.shape}")

    train = train.dropna(subset=history_features).copy()
    print(f"Train after requiring full lag history: {train.shape}")
 
    return train, validation, test

In [7]:
def fit_encoder(train: pd.DataFrame) -> OneHotEncoder:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoder.fit(train[CATEGORICAL_FEATURES])
    return encoder

In [8]:
def build_matrix(
    df: pd.DataFrame,
    encoder: OneHotEncoder,
    numerical_features: list[str],
) -> np.ndarray:
    cat = encoder.transform(df[CATEGORICAL_FEATURES])
    num = df[numerical_features].to_numpy()
    return np.hstack([cat, num])

def _score(y_true, y_pred) -> dict:
    """MAE, MSE, RMSE, MAPE, R2 for one (y_true, y_pred) pair."""
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

In [9]:
def train_all_horizons(
    train, validation, test, X_train, X_validation, X_test
) -> tuple[dict, pd.DataFrame]:
    models = {}
    metrics_rows = []
 
    for h in HORIZONS:
        target_col = f"target_{h}d"
 
        y_train = train[target_col]
        y_val = validation[target_col]
        y_test = test[target_col]
 
        train_mask = y_train.notna().to_numpy()
        val_mask = y_val.notna().to_numpy()
        test_mask = y_test.notna().to_numpy()
 
        model = xgb.XGBRegressor(**XGB_PARAMS)
        model.fit(
            X_train[train_mask],
            y_train[train_mask],
            eval_set=[(X_validation[val_mask], y_val[val_mask])],
            verbose=False,
        )
        models[h] = model
 
        train_pred = model.predict(X_train[train_mask])
        train_scores = _score(y_train[train_mask], train_pred)
 
        val_pred = model.predict(X_validation[val_mask])
        val_scores = _score(y_val[val_mask], val_pred)
 
        if test_mask.sum() > 0:
            test_pred = model.predict(X_test[test_mask])
            test_scores = _score(y_test[test_mask], test_pred)
        else:
            test_scores = {k: np.nan for k in ("MAE", "MSE", "RMSE", "MAPE", "R2")}
 
        row = {
            "horizon": f"{h}-day",
            "train_rows": int(train_mask.sum()),
            "val_rows": int(val_mask.sum()),
            "test_rows": int(test_mask.sum()),
        }
        for split_name, scores in (
            ("train", train_scores),
            ("val", val_scores),
            ("test", test_scores),
        ):
            for metric_name, value in scores.items():
                row[f"{split_name}_{metric_name}"] = value
 
        metrics_rows.append(row)
 
    assert set(models.keys()) == set(HORIZONS), (
        f"Expected a trained model for every horizon {HORIZONS}, "
        f"but only have {sorted(models.keys())}."
    )
 
    results = pd.DataFrame(metrics_rows)
 
    print("\nTRAIN / VALIDATION / TEST RESULTS")
    print("=" * 100)
    for split in ("train", "val", "test"):
        cols = ["horizon", f"{split}_rows"] + [
            f"{split}_{m}" for m in ("MAE", "MSE", "RMSE", "MAPE", "R2")
        ]
        print(f"\n-- {split.upper()} --")
        print(results[cols].to_string(index=False))
 
    for row in metrics_rows:
        if row["test_rows"] == 0:
            print(
                f"\nNote: {row['horizon']} has no test rows with a valid target "
                "yet (targets that far out haven't arrived in the test window)."
            )
 
    return models, results

In [10]:
def get_feature_importance(
    models: dict,
    encoder: OneHotEncoder,
    numerical_features: list[str],
) -> pd.DataFrame:
    feature_names = list(encoder.get_feature_names_out(CATEGORICAL_FEATURES)) + numerical_features
 
    rows = []
    for h, model in models.items():
        importances = model.feature_importances_
        for name, importance in zip(feature_names, importances):
            rows.append({"horizon": f"{h}-day", "feature": name, "importance": importance})
 
    importance_df = pd.DataFrame(rows)
 
    print("\nTOP 10 FEATURES PER HORIZON")
    print("=" * 70)
    for h in HORIZONS:
        top = (
            importance_df[importance_df["horizon"] == f"{h}-day"]
            .sort_values("importance", ascending=False)
            .head(10)
        )
        print(f"\n-- {h}-day --")
        print(top[["feature", "importance"]].to_string(index=False))
 
    return importance_df

In [11]:
def forecast_latest(
    model_data: pd.DataFrame,
    history_features: list[str],
    numerical_features: list[str],
    encoder: OneHotEncoder,
    models: dict,
) -> pd.DataFrame:
    latest = (
        model_data.sort_values(["route_id", "date"])
        .groupby("route_id")
        .tail(1)
        .copy()
    )
 
    usable = latest.dropna(subset=history_features).copy()
    skipped = len(latest) - len(usable)
    if skipped:
        print(f"\nSkipping {skipped} route(s) without full lag history for forecasting.")
 
    X_latest = build_matrix(usable, encoder, numerical_features)
 
    forecast = usable[["route_id", "date", "freight_usd_mt"]].copy()
    forecast = forecast.rename(
        columns={"date": "as_of_date", "freight_usd_mt": "latest_freight_usd_mt"}
    )
    for h in HORIZONS:
        forecast[f"pred_{h}d_freight_usd_mt"] = models[h].predict(X_latest)
 
    print("\nFORWARD FORECAST (per route, from latest available data)")
    print("=" * 70)
    print(forecast.to_string(index=False))
 
    return forecast

In [12]:
def save_artifact(
    models: dict,
    encoder: OneHotEncoder,
    numerical_features: list[str],
    history_features: list[str],
    output_path: Path = Path("freight_rate_models.pkl"),
) -> Path:
    artifact = {
        "models": models,
        "encoder": encoder,
        "categorical_features": CATEGORICAL_FEATURES,
        "numerical_features": numerical_features,
        "history_features": history_features,
        "horizons": HORIZONS,
        "lags": LAGS,
        "rolling_windows": ROLLING_WINDOWS,
    }
    with open(output_path, "wb") as f:
        pickle.dump(artifact, f)
    print(f"\nSaved models + encoder -> {output_path.resolve()}")
    return output_path
 
 
def main():
    paths = get_project_paths()
 
    freight = load_data()
    model_data, history_features, numerical_features = engineer_features(freight)
    train, validation, test = split_train_val_test(model_data, history_features)
 
    try:
        processed_path = paths["processed"] / "model_data.parquet"
        model_data.to_parquet(processed_path, index=False)
    except ImportError:

        processed_path = paths["processed"] / "model_data.csv"
        model_data.to_csv(processed_path, index=False)
    print(f"Saved engineered dataset -> {processed_path}")
 
    encoder = fit_encoder(train)
    X_train = build_matrix(train, encoder, numerical_features)
    X_validation = build_matrix(validation, encoder, numerical_features)
    X_test = build_matrix(test, encoder, numerical_features)
    print(
        f"X_train: {X_train.shape} | "
        f"X_validation: {X_validation.shape} | "
        f"X_test: {X_test.shape}"
    )
 
    models, results = train_all_horizons(
        train, validation, test, X_train, X_validation, X_test
    )
    importance_df = get_feature_importance(models, encoder, numerical_features)
    forecast = forecast_latest(
        model_data, history_features, numerical_features, encoder, models
    )
 
    results_path = paths["results"] / "metrics_train_val_test.csv"
    results.to_csv(results_path, index=False)
    print(f"Saved metrics -> {results_path}")
 
    importance_path = paths["results"] / "feature_importance.csv"
    importance_df.to_csv(importance_path, index=False)
    print(f"Saved feature importance -> {importance_path}")
 
    forecast_path = paths["results"] / "forecast_latest.csv"
    forecast.to_csv(forecast_path, index=False)
    print(f"Saved forward forecast -> {forecast_path}")
 
    output_path = save_artifact(
        models,
        encoder,
        numerical_features,
        history_features,
        output_path=paths["models"] / "freight_rate_models.pkl",
    )
 
    return {
        "models": models,
        "encoder": encoder,
        "results": results,
        "importance": importance_df,
        "forecast": forecast,
        "output_path": output_path,
    }

In [13]:
if __name__ == "__main__":
    main()

Freight data: (64860, 12) | 30 routes | 2020-01-01 -> 2025-12-01
Train: (54810, 27) | Validation: (8190, 27) | Test: (1860, 27)
Train after requiring full lag history: (52110, 27)
Saved engineered dataset -> D:\AI-FREIGHT\data\processed\model_data.csv
X_train: (52110, 57) | X_validation: (8190, 57) | X_test: (1860, 57)

TRAIN / VALIDATION / TEST RESULTS

-- TRAIN --
horizon  train_rows  train_MAE  train_MSE  train_RMSE  train_MAPE  train_R2
  7-day       52110   0.099235   0.017200    0.131149    0.004369  0.999902
 30-day       52110   0.230218   0.090832    0.301383    0.010373  0.999483
 60-day       52110   0.223741   0.083745    0.289388    0.010029  0.999523
 90-day       52110   0.217671   0.079850    0.282578    0.009729  0.999545

-- VAL --
horizon  val_rows  val_MAE  val_MSE  val_RMSE  val_MAPE   val_R2
  7-day      8190 0.275786 0.175517  0.418947  0.011615 0.998886
 30-day      8190 1.094196 2.304039  1.517906  0.046892 0.985020
 60-day      8190 1.262260 2.689986  1.640118